# DisattentionFormer — Gutenberg 85M Training

Trains three ~85M-parameter models on Project Gutenberg (English):
- **V1**: DisattentionFormer (word-level tokenization)
- **V2**: DisattentionFormer + Titans Neural Memory (word-level)
- **Baseline**: Standard Transformer (BPE / tiktoken)

**Runtime**: Use GPU (T4/V100/A100). Edit → Notebook settings → GPU.

## 0. Setup

In [ ]:
# Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/joyce_gutenberg'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')

In [ ]:
# Clone the repo
!git clone https://github.com/fermino/joyce.git /content/joyce 2>/dev/null || (cd /content/joyce && git pull)
%cd /content/joyce

In [ ]:
# Install dependencies
!pip install -q torch einops tiktoken sentence-transformers datasets

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Go to Edit → Notebook settings → GPU')

## 1. Download & Prepare Data

Downloads English texts from `manu/project_gutenberg` on HuggingFace,
then tokenizes into word-level (V1/V2) and BPE (Baseline) binary files.

In [ ]:
import re
import json
import numpy as np
from pathlib import Path
from collections import Counter
from datasets import load_dataset

# --- Config ---
OUT_DIR = Path('data/gutenberg')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_TEXTS = 5000       # ~2GB of text
MAX_VOCAB = 80_000
MIN_FREQ = 5
TRAIN_RATIO = 0.95

PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
EOT_TOKEN = '<eot>'
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, EOT_TOKEN]

def tokenize_words(text):
    return re.findall(r"[A-Za-z'\u2019-]+|[0-9]+|[^\s]", text)

# Check if data already exists
if (OUT_DIR / 'word_train.bin').exists() and (OUT_DIR / 'bpe_train.bin').exists():
    print('Data already prepared! Skipping download.')
else:
    print(f'Downloading manu/project_gutenberg (English, first {MAX_TEXTS} texts)...')
    ds = load_dataset('manu/project_gutenberg', split='en', streaming=True)

    texts = []
    total_chars = 0
    for i, example in enumerate(ds):
        text = example.get('text', '')
        if not text or len(text.strip()) < 100:
            continue
        texts.append(text.strip())
        total_chars += len(text)
        if len(texts) % 500 == 0:
            print(f'  {len(texts)} texts, {total_chars/1e6:.0f}M chars...', flush=True)
        if len(texts) >= MAX_TEXTS:
            break

    print(f'Downloaded {len(texts)} texts, {total_chars/1e6:.0f}M chars')

    # --- Build word vocab ---
    print('\nBuilding word vocabulary...')
    counter = Counter()
    for t in texts:
        counter.update(tokenize_words(t))

    filtered = [(w, c) for w, c in counter.items() if c >= MIN_FREQ]
    filtered.sort(key=lambda x: -x[1])
    if len(filtered) > MAX_VOCAB:
        filtered = filtered[:MAX_VOCAB]

    idx2word = list(SPECIAL_TOKENS) + [w for w, _ in filtered]
    word2idx = {w: i for i, w in enumerate(idx2word)}
    print(f'Vocabulary: {len(idx2word):,} tokens')

    vocab_path = OUT_DIR / 'word_vocab.txt'
    vocab_path.write_text('\n'.join(idx2word), encoding='utf-8')

    # --- Word-level tokenization ---
    print('\nTokenizing (word-level)...')
    unk_id = word2idx[UNK_TOKEN]
    eot_id = word2idx[EOT_TOKEN]
    all_word_ids = []
    for t in texts:
        ids = [word2idx.get(w, unk_id) for w in tokenize_words(t)]
        all_word_ids.extend(ids)
        all_word_ids.append(eot_id)
    word_tokens = np.array(all_word_ids, dtype=np.int32)
    print(f'  {len(word_tokens):,} tokens')

    split = int(len(word_tokens) * TRAIN_RATIO)
    word_tokens[:split].tofile(OUT_DIR / 'word_train.bin')
    word_tokens[split:].tofile(OUT_DIR / 'word_val.bin')
    print(f'  Train: {split:,}, Val: {len(word_tokens)-split:,}')
    del word_tokens, all_word_ids

    # --- BPE tokenization ---
    print('\nTokenizing (BPE / tiktoken)...')
    import tiktoken
    enc = tiktoken.get_encoding('gpt2')
    bpe_eot = enc.encode('<|endoftext|>', allowed_special={'<|endoftext|>'})[0]
    all_bpe_ids = []
    for t in texts:
        ids = enc.encode(t)
        all_bpe_ids.extend(ids)
        all_bpe_ids.append(bpe_eot)
    bpe_tokens = np.array(all_bpe_ids, dtype=np.int32)
    print(f'  {len(bpe_tokens):,} tokens')

    split = int(len(bpe_tokens) * TRAIN_RATIO)
    bpe_tokens[:split].tofile(OUT_DIR / 'bpe_train.bin')
    bpe_tokens[split:].tofile(OUT_DIR / 'bpe_val.bin')
    print(f'  Train: {split:,}, Val: {len(bpe_tokens)-split:,}')
    del bpe_tokens, all_bpe_ids, texts

    print('\nData preparation complete!')

In [ ]:
# Verify data files
for f in sorted(Path('data/gutenberg').iterdir()):
    if f.suffix in ('.bin', '.txt'):
        print(f'  {f.name}: {f.stat().st_size/1e6:.1f} MB')

## 2. Build Archetype Tensors

Required for V1 and V2 (not Baseline). Uses sentence-transformers to
encode Jungian archetype passages into d=512 covariance matrices.

In [ ]:
from build_archetypes import ARCHETYPES, build_archetype_tensors
import torch

TENSOR_PATH = 'archetype_tensors_512.pt'
if os.path.exists(TENSOR_PATH):
    print(f'Loading from {TENSOR_PATH}')
    arch_tensors = torch.load(TENSOR_PATH, map_location='cpu', weights_only=True)
else:
    print('Building archetype tensors (d=512)...')
    arch_tensors = build_archetype_tensors(ARCHETYPES, target_dim=512, device='cpu')
    torch.save(arch_tensors, TENSOR_PATH)
    print(f'Saved: {TENSOR_PATH}')

print(f'{len(arch_tensors)} archetypes, shape={list(arch_tensors.values())[0].shape}')

## 3. Train V1 — DisattentionFormer (85M, word-level)

In [ ]:
import math, time, torch, torch.optim as optim
from gutenberg_data import build_gutenberg_dataloader, load_word_tokenizer
from desatencao_model import DisattentionFormer
from desatencao_loss import disattention_loss

device = torch.device('cuda')

# --- Config ---
CFG_V1 = dict(
    d_model=512, n_heads=8, n_layers=8, d_ff=2048, seq_len=512, dropout=0.1,
    epochs=3, batch_size=16, lr=3e-4, weight_decay=0.1, warmup_steps=4000,
    grad_clip=1.0, alpha=0.10, beta=0.05, gamma=0.02,
    log_every=100, save_every=5000, checkpoint_dir='checkpoints_gutenberg_v1',
)

# --- Load data ---
tokenizer = load_word_tokenizer()
CFG_V1['vocab_size'] = tokenizer.vocab_size
train_loader = build_gutenberg_dataloader('train', 'word', CFG_V1['seq_len'], CFG_V1['batch_size'])
val_loader = build_gutenberg_dataloader('val', 'word', CFG_V1['seq_len'], CFG_V1['batch_size'])
print(f'Train batches/epoch: {len(train_loader):,}')

# --- Model ---
at = {k: v.to(device) for k, v in arch_tensors.items()}
model_v1 = DisattentionFormer(
    vocab_size=CFG_V1['vocab_size'], d_model=512, n_heads=8, n_layers=8, d_ff=2048,
    max_seq_len=512, archetype_tensors=at, dropout=0.1,
).to(device)
n_params = sum(p.numel() for p in model_v1.parameters())
print(f'V1 parameters: {n_params:,} ({n_params/1e6:.1f}M)')

# --- Optimizer ---
decay = [p for _, p in model_v1.named_parameters() if p.requires_grad and p.dim() >= 2]
no_decay = [p for _, p in model_v1.named_parameters() if p.requires_grad and p.dim() < 2]
optimizer = optim.AdamW([{'params': decay, 'weight_decay': 0.1}, {'params': no_decay, 'weight_decay': 0.0}],
                        lr=CFG_V1['lr'], betas=(0.9, 0.95))
total_steps = len(train_loader) * CFG_V1['epochs']

def lr_lambda(step):
    if step < CFG_V1['warmup_steps']: return step / max(CFG_V1['warmup_steps'], 1)
    progress = (step - CFG_V1['warmup_steps']) / max(total_steps - CFG_V1['warmup_steps'], 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
os.makedirs(CFG_V1['checkpoint_dir'], exist_ok=True)
arch_names = list(arch_tensors.keys())

print(f'\nTraining V1: {CFG_V1["epochs"]} epochs, {total_steps:,} steps')

In [ ]:
# --- V1 Training Loop ---
global_step = 0
for epoch in range(CFG_V1['epochs']):
    model_v1.train()
    epoch_loss, epoch_steps = 0.0, 0
    t0 = time.time()

    for input_ids, targets in train_loader:
        input_ids, targets = input_ids.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)

        out = model_v1(input_ids, return_internals=True)
        losses = disattention_loss(
            logits=out['logits'], targets=targets, model=model_v1,
            x_prefnn=out['x_prefnn'], archetype_weights=out['archetype_weights'],
            alpha=CFG_V1['alpha'], beta=CFG_V1['beta'], gamma=CFG_V1['gamma'],
        )
        if torch.isnan(losses['loss']): optimizer.zero_grad(set_to_none=True); continue

        losses['loss'].backward()
        torch.nn.utils.clip_grad_norm_(model_v1.parameters(), CFG_V1['grad_clip'])
        optimizer.step()
        scheduler.step()

        global_step += 1
        epoch_loss += losses['loss'].item()
        epoch_steps += 1

        if global_step % CFG_V1['log_every'] == 0:
            w = out['archetype_weights'].detach().mean(0)
            dom = arch_names[w.argmax().item()]
            lr = scheduler.get_last_lr()[0]
            s_per_s = epoch_steps / max(time.time() - t0, 1e-6)
            print(f'  ep={epoch} step={global_step:>7d} | loss={losses["loss"].item():.4f} '
                  f'ce={losses["l_ce"].item():.4f} | arch={dom} | lr={lr:.2e} | {s_per_s:.1f} steps/s',
                  flush=True)

        if global_step % CFG_V1['save_every'] == 0:
            path = os.path.join(CFG_V1['checkpoint_dir'], f'step_{global_step}.pt')
            torch.save({'step': global_step, 'epoch': epoch, 'model': model_v1.state_dict(),
                        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                        'config': CFG_V1}, path)
            # Copy to Drive
            drive_path = os.path.join(DRIVE_DIR, 'v1', f'step_{global_step}.pt')
            os.makedirs(os.path.dirname(drive_path), exist_ok=True)
            !cp {path} {drive_path}
            print(f'  >> Saved: {path} + Drive')

    avg = epoch_loss / max(epoch_steps, 1)
    print(f'\n  Epoch {epoch}: avg_loss={avg:.4f}, time={time.time()-t0:.0f}s')

    # Validation
    model_v1.eval()
    vloss, vsteps = 0.0, 0
    with torch.no_grad():
        for xi, xt in val_loader:
            xi, xt = xi.to(device), xt.to(device)
            out = model_v1(xi)
            vloss += torch.nn.functional.cross_entropy(out['logits'].view(-1, CFG_V1['vocab_size']), xt.view(-1)).item()
            vsteps += 1
    print(f'  Val CE: {vloss/max(vsteps,1):.4f}\n')

# Save final
final_path = os.path.join(CFG_V1['checkpoint_dir'], 'final.pt')
torch.save({'step': global_step, 'epoch': CFG_V1['epochs'],
            'model': model_v1.state_dict(), 'config': CFG_V1}, final_path)
!cp {final_path} {DRIVE_DIR}/v1/final.pt
print(f'V1 complete: {final_path}')

# Free memory
del model_v1, optimizer, scheduler
torch.cuda.empty_cache()

## 4. Train V2 — DisattentionFormer + Titans (85M, word-level)

In [ ]:
from disattention_v2_model import DisattentionFormerV2

CFG_V2 = dict(
    d_model=512, n_heads=8, n_layers=7, d_ff=2048, seq_len=512, dropout=0.1,
    chunk_size=4, memory_depth=2,
    epochs=3, batch_size=16, lr=3e-4, weight_decay=0.1, warmup_steps=4000,
    grad_clip=1.0, alpha=0.10, beta=0.05, gamma=0.02,
    log_every=100, save_every=5000, checkpoint_dir='checkpoints_gutenberg_v2',
    vocab_size=tokenizer.vocab_size,
)

# V2 uses torch.no_grad() (not inference_mode) because Titans memory
# needs autograd.grad inside @torch.enable_grad() blocks
at = {k: v.to(device) for k, v in arch_tensors.items()}
model_v2 = DisattentionFormerV2(
    vocab_size=CFG_V2['vocab_size'], d_model=512, n_heads=8, n_layers=7, d_ff=2048,
    max_seq_len=512, archetype_tensors=at, chunk_size=4, memory_depth=2, dropout=0.1,
).to(device)
n_params = sum(p.numel() for p in model_v2.parameters())
print(f'V2 parameters: {n_params:,} ({n_params/1e6:.1f}M)')

decay = [p for _, p in model_v2.named_parameters() if p.requires_grad and p.dim() >= 2]
no_decay = [p for _, p in model_v2.named_parameters() if p.requires_grad and p.dim() < 2]
optimizer = optim.AdamW([{'params': decay, 'weight_decay': 0.1}, {'params': no_decay, 'weight_decay': 0.0}],
                        lr=CFG_V2['lr'], betas=(0.9, 0.95))
total_steps = len(train_loader) * CFG_V2['epochs']

def lr_lambda_v2(step):
    if step < CFG_V2['warmup_steps']: return step / max(CFG_V2['warmup_steps'], 1)
    progress = (step - CFG_V2['warmup_steps']) / max(total_steps - CFG_V2['warmup_steps'], 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda_v2)
os.makedirs(CFG_V2['checkpoint_dir'], exist_ok=True)
print(f'\nTraining V2: {CFG_V2["epochs"]} epochs, {total_steps:,} steps')

In [ ]:
# --- V2 Training Loop ---
global_step = 0
for epoch in range(CFG_V2['epochs']):
    model_v2.train()
    epoch_loss, epoch_steps = 0.0, 0
    t0 = time.time()

    for input_ids, targets in train_loader:
        input_ids, targets = input_ids.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)

        out = model_v2(input_ids, return_internals=True)
        losses = disattention_loss(
            logits=out['logits'], targets=targets, model=model_v2,
            x_prefnn=out['x_prefnn'], archetype_weights=out['archetype_weights'],
            alpha=CFG_V2['alpha'], beta=CFG_V2['beta'], gamma=CFG_V2['gamma'],
        )
        if torch.isnan(losses['loss']): optimizer.zero_grad(set_to_none=True); continue

        losses['loss'].backward()
        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), CFG_V2['grad_clip'])
        optimizer.step()
        scheduler.step()

        global_step += 1
        epoch_loss += losses['loss'].item()
        epoch_steps += 1

        if global_step % CFG_V2['log_every'] == 0:
            w = out['archetype_weights'].detach().mean(0)
            dom = arch_names[w.argmax().item()]
            lr = scheduler.get_last_lr()[0]
            s_per_s = epoch_steps / max(time.time() - t0, 1e-6)
            print(f'  ep={epoch} step={global_step:>7d} | loss={losses["loss"].item():.4f} '
                  f'ce={losses["l_ce"].item():.4f} | arch={dom} | lr={lr:.2e} | {s_per_s:.1f} steps/s',
                  flush=True)

        if global_step % CFG_V2['save_every'] == 0:
            path = os.path.join(CFG_V2['checkpoint_dir'], f'step_{global_step}.pt')
            torch.save({'step': global_step, 'epoch': epoch, 'model': model_v2.state_dict(),
                        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                        'config': CFG_V2}, path)
            drive_path = os.path.join(DRIVE_DIR, 'v2', f'step_{global_step}.pt')
            os.makedirs(os.path.dirname(drive_path), exist_ok=True)
            !cp {path} {drive_path}
            print(f'  >> Saved: {path} + Drive')

    avg = epoch_loss / max(epoch_steps, 1)
    print(f'\n  Epoch {epoch}: avg_loss={avg:.4f}, time={time.time()-t0:.0f}s')

    model_v2.eval()
    vloss, vsteps = 0.0, 0
    with torch.no_grad():
        for xi, xt in val_loader:
            xi, xt = xi.to(device), xt.to(device)
            out = model_v2(xi)
            vloss += torch.nn.functional.cross_entropy(out['logits'].view(-1, CFG_V2['vocab_size']), xt.view(-1)).item()
            vsteps += 1
    print(f'  Val CE: {vloss/max(vsteps,1):.4f}\n')

final_path = os.path.join(CFG_V2['checkpoint_dir'], 'final.pt')
torch.save({'step': global_step, 'epoch': CFG_V2['epochs'],
            'model': model_v2.state_dict(), 'config': CFG_V2}, final_path)
!mkdir -p {DRIVE_DIR}/v2 && cp {final_path} {DRIVE_DIR}/v2/final.pt
print(f'V2 complete: {final_path}')

del model_v2, optimizer, scheduler
torch.cuda.empty_cache()

## 5. Train Baseline — Standard Transformer (85M, BPE)

In [ ]:
import torch.nn.functional as F
from baseline_model import BaselineTransformer
from gutenberg_data import build_gutenberg_dataloader, load_bpe_tokenizer

CFG_BL = dict(
    d_model=512, n_heads=8, n_layers=19, d_ff=2048, seq_len=512, dropout=0.1,
    vocab_size=50257,
    epochs=3, batch_size=16, lr=3e-4, weight_decay=0.1, warmup_steps=4000,
    grad_clip=1.0, log_every=100, save_every=5000,
    checkpoint_dir='checkpoints_gutenberg_baseline',
)

bpe_tokenizer = load_bpe_tokenizer()
bl_train_loader = build_gutenberg_dataloader('train', 'bpe', CFG_BL['seq_len'], CFG_BL['batch_size'])
bl_val_loader = build_gutenberg_dataloader('val', 'bpe', CFG_BL['seq_len'], CFG_BL['batch_size'])
print(f'Train batches/epoch: {len(bl_train_loader):,}')

model_bl = BaselineTransformer(
    vocab_size=50257, d_model=512, n_heads=8, n_layers=19, d_ff=2048,
    max_seq_len=512, dropout=0.1,
).to(device)
n_params = sum(p.numel() for p in model_bl.parameters())
print(f'Baseline parameters: {n_params:,} ({n_params/1e6:.1f}M)')

decay = [p for _, p in model_bl.named_parameters() if p.requires_grad and p.dim() >= 2]
no_decay = [p for _, p in model_bl.named_parameters() if p.requires_grad and p.dim() < 2]
optimizer = optim.AdamW([{'params': decay, 'weight_decay': 0.1}, {'params': no_decay, 'weight_decay': 0.0}],
                        lr=CFG_BL['lr'], betas=(0.9, 0.95))
total_steps = len(bl_train_loader) * CFG_BL['epochs']

def lr_lambda_bl(step):
    if step < CFG_BL['warmup_steps']: return step / max(CFG_BL['warmup_steps'], 1)
    progress = (step - CFG_BL['warmup_steps']) / max(total_steps - CFG_BL['warmup_steps'], 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda_bl)
os.makedirs(CFG_BL['checkpoint_dir'], exist_ok=True)
print(f'\nTraining Baseline: {CFG_BL["epochs"]} epochs, {total_steps:,} steps')

In [ ]:
# --- Baseline Training Loop ---
global_step = 0
for epoch in range(CFG_BL['epochs']):
    model_bl.train()
    epoch_loss, epoch_steps = 0.0, 0
    t0 = time.time()

    for input_ids, targets in bl_train_loader:
        input_ids, targets = input_ids.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)

        out = model_bl(input_ids)
        loss = F.cross_entropy(out['logits'].view(-1, 50257), targets.view(-1))
        if torch.isnan(loss): optimizer.zero_grad(set_to_none=True); continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_bl.parameters(), CFG_BL['grad_clip'])
        optimizer.step()
        scheduler.step()

        global_step += 1
        epoch_loss += loss.item()
        epoch_steps += 1

        if global_step % CFG_BL['log_every'] == 0:
            lr = scheduler.get_last_lr()[0]
            s_per_s = epoch_steps / max(time.time() - t0, 1e-6)
            print(f'  ep={epoch} step={global_step:>7d} | loss={loss.item():.4f} | '
                  f'lr={lr:.2e} | {s_per_s:.1f} steps/s', flush=True)

        if global_step % CFG_BL['save_every'] == 0:
            path = os.path.join(CFG_BL['checkpoint_dir'], f'step_{global_step}.pt')
            torch.save({'step': global_step, 'epoch': epoch, 'model': model_bl.state_dict(),
                        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                        'config': CFG_BL}, path)
            drive_path = os.path.join(DRIVE_DIR, 'baseline', f'step_{global_step}.pt')
            os.makedirs(os.path.dirname(drive_path), exist_ok=True)
            !cp {path} {drive_path}
            print(f'  >> Saved: {path} + Drive')

    avg = epoch_loss / max(epoch_steps, 1)
    print(f'\n  Epoch {epoch}: avg_loss={avg:.4f}, time={time.time()-t0:.0f}s')

    model_bl.eval()
    vloss, vsteps = 0.0, 0
    with torch.no_grad():
        for xi, xt in bl_val_loader:
            xi, xt = xi.to(device), xt.to(device)
            out = model_bl(xi)
            vloss += F.cross_entropy(out['logits'].view(-1, 50257), xt.view(-1)).item()
            vsteps += 1
    print(f'  Val CE: {vloss/max(vsteps,1):.4f}\n')

final_path = os.path.join(CFG_BL['checkpoint_dir'], 'final.pt')
torch.save({'step': global_step, 'epoch': CFG_BL['epochs'],
            'model': model_bl.state_dict(), 'config': CFG_BL}, final_path)
!mkdir -p {DRIVE_DIR}/baseline && cp {final_path} {DRIVE_DIR}/baseline/final.pt
print(f'Baseline complete: {final_path}')

del model_bl, optimizer, scheduler
torch.cuda.empty_cache()

## 6. Summary

All checkpoints saved to both local storage and Google Drive at:
- `drive/MyDrive/joyce_gutenberg/v1/`
- `drive/MyDrive/joyce_gutenberg/v2/`
- `drive/MyDrive/joyce_gutenberg/baseline/`

Download the `final.pt` files to your local `joyce/` repo for benchmarking.

In [ ]:
# List all checkpoint files saved to Drive
import os
for root, dirs, files in os.walk(DRIVE_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f'  {path}: {size_mb:.1f} MB')